### Relation Extraction
The model bert-base-uncased is used from huggingface as pre-trained model

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import preprocessor
from preprocessor import Preprocessor

In [41]:
ROOT_DIR = preprocessor.ROOT_DIR
DATA_PATH = preprocessor.DATA_PATH

preprocessor = Preprocessor(ROOT_DIR)

In [51]:
data = preprocessor.loadFile('final_ass1_data.json')
# 122 instances in final_ass1_data.json
print(data[0])

{'id': 70476117, 'annotations': [{'id': 22591087, 'completed_by': {'id': 12716, 'email': 'f.a.ensink.op.kemma@student.tue.nl', 'first_name': '', 'last_name': ''}, 'result': [{'id': 'a-cF8klU4-', 'type': 'labels', 'value': {'end': 8, 'text': 'Ephesus', 'start': 1, 'labels': ['landmark_name']}, 'origin': 'manual', 'to_name': 'text', 'from_name': 'label'}, {'id': 'fUvEjMgG3v', 'type': 'labels', 'value': {'end': 73, 'text': 'Ancient Greece', 'start': 59, 'labels': ['location']}, 'origin': 'manual', 'to_name': 'text', 'from_name': 'label'}, {'id': 'lg86acnG1f', 'type': 'labels', 'value': {'end': 127, 'text': 'Seluk', 'start': 122, 'labels': ['location']}, 'origin': 'manual', 'to_name': 'text', 'from_name': 'label'}, {'id': 'WOx1LV2UFN', 'type': 'labels', 'value': {'end': 144, 'text': 'zmir Province', 'start': 131, 'labels': ['location']}, 'origin': 'manual', 'to_name': 'text', 'from_name': 'label'}, {'id': 'laDHrMHSe0', 'type': 'labels', 'value': {'end': 152, 'text': 'Turkey', 'start': 146,

In [43]:
# Extract text data and entity annotations
text_data = [item['annotations'][0]['result'][0]['value']['text'] for item in data]
entity_annotations = [item['annotations'][0]['result'] for item in data]


text_data[0]            -> title name of wikipedia  ===> "Ephesus"

entity_annotations[0]   ->                          ===> "[{'id': 'a-cF8klU4-', 'type': 'labels', 'value': {'end': 8, 'text': 'Ephesus', 'start': 1, 'labels': ['landmark_name']}, 'origin': 'manual', 'to_name': 'text',"

tokenized_data.keys() ===> dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])
print(tokenized_data['input_ids'].shape) ===> torch.Size([122, 17])

In [44]:
# Tokenize the text data
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
tokenized_data = tokenizer(text_data, padding=True, truncation=True, return_tensors="pt")

In [45]:
print(tokenized_data['attention_mask'].shape)

torch.Size([122, 17])


In [46]:
# Define relation labels
relation_labels = {
    "org:created_by": 0,
    "org:happened_on": 1,
    "org:located_in": 2,
    "org:has_occupation": 3,
    "org:is_condition": 4,
    "org:enlisted_in": 5,
    "org:is_type": 6,
    "org:has_component": 7,
    "org:is_similar_to": 8
}

# Extract and prepare relation labels based on your data
y = []
for annotations in entity_annotations:
    relations = [annotation for annotation in annotations if annotation['type'] == 'relation']
    
    # Check if there are any relations for this data point
    if relations:
        relation_labels_for_data = []
        
        for relation in relations:
            if 'labels' in relation and relation['labels']:
                relation_labels_for_data.append(relation_labels.get(relation['labels'][0], -1))
            else:
                # Handle cases where the relation does not have labels (assign a default label, ignore, or handle as needed)
                relation_labels_for_data.append(-1)
        
        # Append the relation label (you may need to decide how to handle multiple relations)
        if relation_labels_for_data:
            y.append(relation_labels_for_data[0])
        else:
            # Handle cases where there are no labeled relations (assign a default label, ignore, or handle as needed)
            y.append(-1)
    else:
        # Handle cases where there are no relations at all (assign a default label, ignore, or handle as needed)
        y.append(-1)

# Filter out instances with unknown relation labels (-1)
X_train_tokenized = [text for i, text in enumerate(tokenized_data) if y[i] != -1]
y = [label for label in y if label != -1]


In [47]:
print(len(X_train_tokenized))
print(len(y))

3
121


In [48]:
# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_train_tokenized, y, test_size=0.2, random_state=42)

# Continue with the training and evaluation code provided earlier

ValueError: Found input variables with inconsistent numbers of samples: [3, 121]

In [ ]:
# Load BERT model for sequence classification
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(relation_labels))

# Define optimizer and set hyperparameters
optimizer = AdamW(model.parameters(), lr=1e-5)

# Training loop
num_epochs = 3  # You can adjust the number of epochs
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(**X_train_tokenized, labels=torch.tensor(y_train))
    loss = outputs.loss
    loss.backward()
    optimizer.step()

# Evaluation
model.eval()
with torch.no_grad():
    outputs = model(**X_test_tokenized)
    predicted_labels = outputs.logits.argmax(dim=1)

print(classification_report(y_test, predicted_labels))